In [ ]:
import sys
from pathlib import Path
import importlib

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42
# True: rebuild the encoded split from the raw CSV; False: load its local cache.
# The flag controls input preparation, not model training.
REBUILD = True

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","6g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
df_raw = data_oi.load_raw_data()
df_raw.show(truncate= 80)
print("rows:", df_raw.count())

### Exploratory Data Analysis (EDA)

Begin by trying to comprehend what your dataset conveys.

- What type of data are you currently exploring?
- What is the granularity of the data?
- What does the distribution of the data look like? Are there any outliers? Are there any missing or invalid entries?
- Is there any correlation between the variables you are interested in exploring?
- How would you cleanly and accurately visualize the relationship among variables?

In [ ]:
from src.eda import EDA

eda_raw = EDA(df_raw)
eda_raw.dataframe_overview()

#### Fields

Each row is one comment on a drug, with the condition and a satisfaction score. The fields mix free text, categories, integers, and a date.

| Field | Type | Content |
|---|---|---|
| `uniqueID` | integer | Review identifier. There is no patient identifier. |
| `drugName` | string | Drug name as written. |
| `condition` | string | Condition or indication. |
| `review` | string | Comment text. Some values contain HTML entities such as `&#039;` and surrounding quotation marks. |
| `rating` | integer | Satisfaction score from 1 to 10. |
| `date` | date | Publication date. |
| `usefulCount` | integer | Helpful votes accumulated after publication. |


In [ ]:
eda_raw.granularity()

#### Granularity

The file has **161,297** reviews. `uniqueID` is unique, and the file has no patient identifier, so this is a review count.

- **3,436** distinct `drugName` strings. One comment can appear under more than one name, such as Sertraline and Zoloft.
- **885** distinct `condition` values, including null and text that is not a condition name.
- **48,635** groups share identical `review` text. **72** of those groups contain more than one `rating`.


In [ ]:
eda_raw.distribution_inspect()

#### Distributions and invalid values

`rating` stays inside 1–10. The scores pile up at the ends: **50,989** reviews (31.6%) are 10 and **21,619** (13.4%) are 1, while 2–7 are smaller. The standard deviation is 3.27 and the skewness is −0.80.

`usefulCount` has a long right tail (skewness 4.67). The median is 16, the third quartile is 36, and the maximum is 1,291. That maximum is one comment stored twice, under Sertraline and Zoloft.

Review length has a median of 455 characters and a maximum of 10,745. Six comments are three characters long. Some `condition` labels are truncated, including `Testicular Cance` and `min)`.

| Issue | Rows |
|---|---|
| Null `condition` | 899 (0.56%) |
| Helpful-vote text stored in `condition` | 900 (0.56%) |
| Nulls in the other six fields | 0 |
| Whitespace-only values | 0 |
| `rating` outside the integers 1–10 | 0 |
| Negative or non-numeric `usefulCount` | 0 |
| Dates that fail `d-MMM-yy`, or fall after 2026-11-02 | 0 |

The 900 `condition` values are page fragments such as `2</span> users found this comment helpful.`, separate from the 899 nulls. Comment text also contains HTML entities such as `&#039;`, and some condition names are cut off, such as `Panic Disorde` and `Stomach Ulce`.


In [ ]:
eda_raw.correlation()

#### Correlations

Pearson correlation uses `rating`, `usefulCount`, and `date` as days since 1970-01-01. A larger date value is a later review.

| Pair | r | Reading |
|---|---:|---|
| Rating, helpful votes | 0.234 | Higher ratings tend to receive somewhat more votes. |
| Rating, date | −0.196 | Later reviews have slightly lower ratings. |
| Helpful votes, date | −0.269 | Earlier reviews have more votes, consistent with a longer time to collect them. |

All three absolute values are below 0.3, so the linear associations are weak. `rating` is an ordinal score, and `usefulCount` has a long right tail. Drug and condition names stay categorical and are left out of this calculation.


### Data Cleaning
A crucial step in the data preprocessing pipeline, aimed at improving the quality and accuracy of the dataset before analysis.
- Identifying Missing Values
- Handling Outliers
- Correcting Invalid Entries
- Removing Duplicates
- Standardizing Data: Ensure consistency in data representation.
- Transforming Variables: Apply necessary transformations to variables to enhance their interpretability or suitability for analysis.

In [ ]:
from src.cleaning import data_cleaning

if REBUILD:
    print("Before data cleaning:")
    df_raw.show(10,truncate=80)
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    print("After data cleaning:")
    df_cleaned.show(10,truncate=80)
    print("Cleaned data overview:")
    eda_cleaned = EDA(df_cleaned)
    eda_cleaned.dataframe_overview()
    eda_cleaned.granularity()
    eda_cleaned.distribution_inspect()


In [ ]:
import src.text_features
importlib.reload(src.text_features)
from src.text_features import text_feature_engineering, baseline_text_feature_engineering

df_baseline_engineered = baseline_text_feature_engineering(df_cleaned)
df_baseline_engineered.cache().count()

if REBUILD:
    print("Before text feature engineering:")
    df_cleaned.show(10,truncate = 80)
    df_engineered = text_feature_engineering(df_cleaned,True)
    df_engineered.cache().count()
    print("After text feature engineering:")
    df_engineered.show(10,truncate = 80)


In [ ]:
from src.encoder import encoder
from src.split import split_and_weight

df_baseline_train, df_baseline_test = split_and_weight(df_baseline_engineered,SEED)
df_baseline_train,df_baseline_test = encoder(df_baseline_train,df_baseline_test)

if REBUILD:
    df_train, df_test = split_and_weight(df_engineered, SEED)
    df_train, df_test = encoder(df_train, df_test)
    data_oi.save_split_data(df_train,df_test)
else:
    df_train, df_test = data_oi.load_split_data()

In [ ]:
from src.embeddings import EmbeddingModel
# Embedding Baseline Model (TFIDF without text feature engineering)

embedding_baseline_model = EmbeddingModel(df_baseline_train,df_baseline_test)
baseline_name = "baseline"
print("Embedding Baseline Model `baseline` Training")
embedding_baseline_model.tfidf_train(baseline_name)
print("Embedding Baseline Model `baseline` Running")
train_embedding_baseline, test_embedding_baseline = embedding_baseline_model.tfidf_predict(baseline_name)

In [ ]:
from src.embeddings import EmbeddingModel

# Fit registered embeddings before prediction; downstream models read `embedding`.
embedding_model = EmbeddingModel(df_train,df_test)

for name, train_function in EmbeddingModel.train_registry.items():
    print(f"Embedding Model `{name}` Training")
    train_function(embedding_model)

In [ ]:
train_embedding_dict = {}
test_embedding_dict = {}

for name, predict_function in EmbeddingModel.predict_registry.items():
    print(f"Embedding Model `{name}` Running")
    train_embedding_dict[name], test_embedding_dict[name] = predict_function(embedding_model)
    data_oi.save_embedding_data(name,train_embedding_dict[name],test_embedding_dict[name])

In [ ]:
# Analytics Baseline Model (Logistic Regression Model without weight col)

from src.analytics import AnalyticsModel
analytics_baseline_model = AnalyticsModel(train_embedding_baseline,test_embedding_baseline)

print("Analytics Baseline Model `lr` Training")
analytics_baseline_model.lr_train(baseline_name)
print("Analytics Baseline Model `lr` Running")
train_analytics_baseline, test_analytics_baseline = analytics_baseline_model.lr_predict(baseline_name)

In [ ]:
from src.analytics import AnalyticsModel

train_embedding_dict, test_embedding_dict = data_oi.load_embedding_data()

analytics_models = {}
for embedding_name in train_embedding_dict.keys():
    analytics_models[embedding_name] = AnalyticsModel(
        train_embedding_dict[embedding_name],
        test_embedding_dict[embedding_name]
    )
    for name, train_function in AnalyticsModel.train_registry.items():
        print(f"Embedding Model `{embedding_name}` Analytics Model `{name}` Training")
        train_function(analytics_models[embedding_name], embedding_name)

In [ ]:
train_pred_dict = {}
test_pred_dict = {}
for embedding_name in train_embedding_dict.keys():
    for name, predict_function in AnalyticsModel.predict_registry.items():
        print(f"Embedding Model `{embedding_name}` Analytics Model `{name}` Running")
        # Format: `{analytics}__{embedding}`. Either name may contain a single underscore.
        model_name = f"{name}__{embedding_name}"
        train_pred_dict[model_name], test_pred_dict[model_name] = predict_function(
            analytics_models[embedding_name],
            embedding_name
        )
        data_oi.save_prediction_data(
            model_name,
            train_pred_dict[model_name],
            test_pred_dict[model_name]
        )


In [ ]:
from src.evaluation import Scorer

train_pred_dict, test_pred_dict = data_oi.load_prediction_data()

scorer_train = Scorer(train_pred_dict)
scorer_train.add_model("baseline","lr",train_analytics_baseline)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
scorer_test.add_model("baseline","lr",test_analytics_baseline)
print("Tesing Data:")
scorer_test.show_metrics(truncate = 80)
